In [1]:
import bw2data, bw2io
import bw2calc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os
import sys

In [2]:
sys.path.append('/Users/susierwu/dpLCA_main/') 
from utils import *
from utils.newbw2method_dpLCIA_addBW25 import * 

In [3]:
bw2data.projects.set_current('ei311')
#list(bw2data.databases)
#[m for m in bw2data.methods if 'pGWP100' in str(m) and  'SSP119' in str(m)]

In [4]:
# already have 18 pGWP100, we'll add pGWP20 now
len([m for m in bw2data.methods if 'pGWP' in str(m)])

18

In [5]:
mybio = bw2data.Database("ecoinvent-3.11-biosphere")
len(mybio)

9795

### read in pre-calculated .nc LCIA dataset,  for GWP20, it's important to make sure 
##### the cf_point below dataarray has only filtered years from 1 up to 20, do not use the 100, otherwise the assign_majorghg_dCC() function with ` assert len(cf_touse[ch4_].values ) == len(C["Methane, fossil"])` generate error because the module was initially prepared only for GWP100 with len 100 

In [6]:
cf_point = xr.open_dataset('../../../dpLCIA/AGWPCO2_fixed_IPCCAR6/output_dpGWP_fixedCO2/CF_GWP1_100_perSSP_MY_majorghgs.nc')
#cf_point

In [7]:
cf_point20 = cf_point.where(cf_point["Year"] <= 20, drop=True)

### read in premise_GWP, all minor_ghg using static amount 

In [8]:
prem_gwp20_dfraw = pd.read_excel("../../../dpLCIA/LCIA/premise_gwp/lcia_gwp2021_20a_w_bio.xlsx")
prem_gwp20_dfraw.head()

,name,categories,amount
0,Bromopropane,air::unspecified,0.188
1,Butane,air::urban air close to ground,0.022
2,Butane,air::non-urban air or from high stacks,0.022
3,Butane,"air::low population density, long-term",0.022
4,Butane,air::lower stratosphere + upper troposphere,0.022


### calling the assign_dpGWP class, see what it looks like for final CF

In [9]:
xx = assign_dpGWP(premise_gwp100_inputdf = prem_gwp20_dfraw, 
                  cf_inputds = cf_point20, 
                  TH = 21,  # this is import to change to 21 for GWP20, default was 101 for GWP100
                  ssp = '119', fairMY = 2030 )

minorg, allg = xx.get_minorand_allGHG()
cc = xx.prep_empty_C(allg)
#cc.head()
pcc = xx.assign_minorghg_to_C_GWP100(cc, minorg)
fcc = xx.assign_majorghg_dCC(pcc)
fcc

,Bromopropane,Butane,"Carbon monoxide, fossil","Carbon monoxide, from soil or biomass stock","Carbon monoxide, non-fossil",Chloroform,Ethane,"Ethane, 1,1,1,2-tetrafluoro-, HFC-134a","Ethane, 1,1,1-trichloro-, HCFC-140","Ethane, 1,1,1-trifluoro-, HFC-143a",...,"Carbon dioxide, non-fossil","Carbon dioxide, in air","Carbon dioxide, to soil or biomass stock","Carbon dioxide, from soil or biomass stock","Carbon dioxide, fossil","Carbon dioxide, non-fossil, resource correction","Methane, from soil or biomass stock","Methane, fossil","Methane, non-fossil",Dinitrogen monoxide
GWP1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,129.352601,129.352601,129.352601,184.469938
GWP2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,127.959994,127.959994,127.959994,189.550487
GWP3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,126.326546,126.326546,126.326546,194.249359
GWP4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,124.507922,124.507922,124.507922,198.568767
GWP5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,122.475229,122.475229,122.475229,202.518172
GWP6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,120.353968,120.353968,120.353968,206.092315
GWP7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,118.122406,118.122406,118.122406,209.342129
GWP8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,115.777326,115.777326,115.777326,212.292888
GWP9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,113.428104,113.428104,113.428104,214.969948
GWP10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.144749,-1.144749,-1.144749,1.144749,1.144749,-1.144749,111.048411,111.048411,111.048411,217.397875


### run MY2030/2040&2050 and all SSP together, for GWP we only have pGWP100 now, using static GWP100 for minorGHGs

In [10]:
for mmy in [2030, 2040, 2050]: 
    for sp in ['119', '245', '585']: 

        xx = assign_dpGWP(premise_gwp100_inputdf = prem_gwp20_dfraw, 
                  cf_inputds = cf_point20, 
                  TH = 21,  #  to change to 21 for GWP20, default was 101 for GWP100
                  ssp = sp, fairMY = mmy ) 

        
        minorg, allg = xx.get_minorand_allGHG()
        emt_C =  xx.prep_empty_C(allg)
        fullminor_C = xx.assign_minorghg_to_C_GWP100(emt_C, minorg )
        #print(fullminor_C.head() )
        full_allC = xx.assign_majorghg_dCC(fullminor_C)
        data = xx.prep_data_for_bw2method (full_allC, 
                                           mybio = bw2data.Database("ecoinvent-3.11-biosphere") , 
                                           mybio_str = "ecoinvent-3.11-biosphere")
        print(len(data))
        xx.prep_final_dCC_bw2method ( C = full_allC , data = data, gwp_method = '- fixed-AGWPCO2')

start preparing data to be assigned as new bw2data.methods, for SSP 119 and fair_MY2030
1
start creating new method, under: SSP119, MY2030 
We'll write GWP20 values
finishing preparing new methods, method name : ('Climate Change prospective GWP20', 'SSP119', 'MY2030', 'pGWP20 - fixed-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 245 and fair_MY2030
1
start creating new method, under: SSP245, MY2030 
We'll write GWP20 values
finishing preparing new methods, method name : ('Climate Change prospective GWP20', 'SSP245', 'MY2030', 'pGWP20 - fixed-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 585 and fair_MY2030
1
start creating new method, under: SSP585, MY2030 
We'll write GWP20 values
finishing preparing new methods, method name : ('Climate Change prospective GWP20', 'SSP585', 'MY2030', 'pGWP20 - fixed-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 119 and fair_MY2040
1
start creating new method

In [11]:
[m for m in bw2data.methods if 'pGWP20' in str(m)]

[('Climate Change prospective GWP20',
  'SSP119',
  'MY2030',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP245',
  'MY2030',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP585',
  'MY2030',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP119',
  'MY2040',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP245',
  'MY2040',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP585',
  'MY2040',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP119',
  'MY2050',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP245',
  'MY2050',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP585',
  'MY2050',
  'pGWP20 - fixed-AGWPCO2')]